In [ ]:
from typing import List, Dict, Optional
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import json
import numpy as np
import pandas as pd
import os
import pickle
from datetime import date
from tqdm import tqdm
from dataclasses import dataclass, field
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
from datetime import datetime
import transformers, sklearn
print(torch.__version__, transformers.__version__, sklearn.__version__)

In [ ]:
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")
if cuda_available:
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# NUM_EMOTIONS = 28 # This is for goEmotion
# THRESHOLD    = 0.3

NUM_EMOTIONS = 14
THRESHOLD    = 0.3

# 0. Data Loading

In [ ]:
@dataclass
class SRLFrame:
    predicate_word_idx: int
    labels: List[str]
    predicate_form: Optional[str] = None
    arg_head_idx: Optional[List[int]] = None

@dataclass
class UtteranceSample:
    words:    List[str]
    frames:   List[SRLFrame]
    emotions: Optional[List[int]] = None  # emotion indices (0-27)


def create_utterance_samples(data_path: str) -> List[UtteranceSample]:
    """Load aligned JSONL produced by the dataset creation notebook."""
    samples = []
    with open(data_path, "r", encoding="utf-8") as f:
        for line in f:
            ex = json.loads(line)
            samples.append(UtteranceSample(
                words=ex["words"],
                frames=[SRLFrame(**fr) for fr in ex.get("frames", [])],
                emotions=ex.get("emotions", None),
            ))
    return samples


def emotions_to_vector(emotion_indices: List[int], num_emotions: int = NUM_EMOTIONS) -> torch.Tensor:
    """Convert active emotion indices to a binary multi-hot vector [num_emotions]."""
    vec = torch.zeros(num_emotions, dtype=torch.float)
    for idx in emotion_indices:
        if 0 <= idx < num_emotions:
            vec[idx] = 1.0
    return vec

# 1. SRL Dataset (Utterance Level)

In [ ]:
class SRLDataset(Dataset):
    """
    Multi-frame utterance-level dataset for SRL-Aware models.
    BERT input : [CLS] <utterance tokens> [SEP]  (single segment)
    Target     : 28-dim binary multi-hot emotion vector
    """
    def __init__(self, samples: List[UtteranceSample], tokenizer, label2id: Dict[str, int],
                 max_length: int = 128, debug_print: bool = False):
        self.samples     = samples
        self.tokenizer   = tokenizer
        self.label2id    = label2id
        self.max_length  = max_length
        self.debug_print = debug_print

    def __len__(self): return len(self.samples)

    # def _build_word_first_wp_idx(self, words):
    #     tmp = self.tokenizer(words, is_split_into_words=True)
    #     first_pos = {}
    #     for pos, wid in enumerate(tmp.word_ids()):
    #         if wid is not None and wid not in first_pos:
    #             first_pos[wid] = pos
    #     return [first_pos[w] for w in range(len(words))]
    
    def _build_word_first_wp_idx(self, words):
        tmp = self.tokenizer(
            words,
            is_split_into_words=True,
            add_special_tokens=False,
            return_attention_mask=False,
            return_token_type_ids=False,
            truncation=False,
        )

        first_pos = {}
        for pos, wid in enumerate(tmp.word_ids()):
            if wid is not None and wid not in first_pos:
                first_pos[wid] = pos

        missing = [w for w in range(len(words)) if w not in first_pos]
        if missing and self.debug_print:
            print("Missing word alignments:", missing)
            print("Words:", words)
            print("word_ids():", tmp.word_ids())

        # shift by +1 because [CLS] is prepended later
        return [first_pos[w] + 1 if w in first_pos else -1 for w in range(len(words))]


    @staticmethod
    def _norm(lbl): return lbl.replace("R-", "").replace("C-", "")

    def _frame_tensors(self, words, frame):
        n       = len(words)
        norm_l  = [self._norm(l) for l in frame.labels]
        unk     = self.label2id.get("O", 0)
        lids    = [self.label2id.get(l, unk) for l in norm_l]

        role = [0] * n
        for i, t in enumerate(frame.labels):
            if i == frame.predicate_word_idx: role[i] = 1
            if   "ARG0" in t: role[i] = 2
            elif "ARG1" in t: role[i] = 3
            elif "ARG2" in t: role[i] = 4
            elif "ARGM" in t: role[i] = 5

        out = {
            "labels":        torch.tensor(lids,  dtype=torch.long),
            "pred_word_idx": torch.tensor(frame.predicate_word_idx, dtype=torch.long),
            "role_ids":      torch.tensor(role,  dtype=torch.long),
            "arg0_mask":     torch.tensor([1 if "ARG0" in t else 0 for t in norm_l], dtype=torch.long),
            "arg1_mask":     torch.tensor([1 if "ARG1" in t else 0 for t in norm_l], dtype=torch.long),
            "arg2_mask":     torch.tensor([1 if "ARG2" in t else 0 for t in norm_l], dtype=torch.long),
            "argm_mask":     torch.tensor([1 if "ARGM" in t else 0 for t in norm_l], dtype=torch.long),
        }
        if frame.arg_head_idx is not None:
            out["arg_head_idx"] = torch.tensor(frame.arg_head_idx, dtype=torch.long)
        return out

    def __getitem__(self, idx):
        inst   = self.samples[idx]
        words  = inst.words
        n      = len(words)

        enc    = self.tokenizer(words, is_split_into_words=True, add_special_tokens=False,
                                return_attention_mask=False, return_token_type_ids=False)
        ids    = [self.tokenizer.cls_token_id] + enc["input_ids"] + [self.tokenizer.sep_token_id]
        ttids  = [0] * len(ids)
        amask  = [1] * len(ids)
        wfp    = self._build_word_first_wp_idx(words)

        if len(ids) > self.max_length:
            ids   = ids[:self.max_length]
            ttids = ttids[:self.max_length]
            amask = amask[:self.max_length]
            wfp   = [min(p, self.max_length - 1) for p in wfp]

        frame_dicts = [self._frame_tensors(words, fr) for fr in inst.frames
                       if fr and fr.labels and fr.predicate_word_idx is not None]

        res = {
            "input_ids":             torch.tensor(ids,   dtype=torch.long),
            "token_type_ids":        torch.tensor(ttids, dtype=torch.long),
            "attention_mask":        torch.tensor(amask, dtype=torch.long),
            "word_first_wp_fullidx": torch.tensor(wfp,   dtype=torch.long),
            "sent_len":              torch.tensor(n,      dtype=torch.long),
            "frames":                frame_dicts,
        }
        if inst.emotions is not None:
            res["emotion_labels"] = emotions_to_vector(inst.emotions, num_emotions=NUM_EMOTIONS)
        return res

# 2. Collate Functions

In [ ]:
def srl_collate_ulevel(batch: List[Dict], pad_token_id: int, pad_label_id: int = -100):
    B     = len(batch)
    max_L = max(x["input_ids"].size(0) for x in batch)
    max_n = max(int(x["sent_len"]) for x in batch)
    max_F = max(len(x.get("frames", [])) for x in batch)

    input_ids      = torch.full((B, max_L), pad_token_id, dtype=torch.long)
    token_type_ids = torch.zeros((B, max_L), dtype=torch.long)
    attention_mask = torch.zeros((B, max_L), dtype=torch.long)
    word_first_wp  = torch.full((B, max_n), -1, dtype=torch.long)
    sent_lens      = torch.zeros((B,), dtype=torch.long)
    sentence_mask  = torch.zeros((B, max_n), dtype=torch.bool)

    frames_mask          = torch.zeros((B, max_F), dtype=torch.bool)
    frames_labels        = torch.full((B, max_F, max_n), pad_label_id, dtype=torch.long)
    frames_pred_word_idx = torch.zeros((B, max_F), dtype=torch.long)
    frames_role_ids      = torch.zeros((B, max_F, max_n), dtype=torch.long)
    frames_arg0_mask     = torch.zeros((B, max_F, max_n), dtype=torch.long)
    frames_arg1_mask     = torch.zeros((B, max_F, max_n), dtype=torch.long)
    frames_arg2_mask     = torch.zeros((B, max_F, max_n), dtype=torch.long)
    frames_argm_mask     = torch.zeros((B, max_F, max_n), dtype=torch.long)

    has_emotion = "emotion_labels" in batch[0]
    if has_emotion:
        emotion_labels = torch.zeros((B, NUM_EMOTIONS), dtype=torch.float)

    for i, item in enumerate(batch):
        L = item["input_ids"].size(0)
        input_ids[i, :L]      = item["input_ids"]
        token_type_ids[i, :L] = item["token_type_ids"]
        attention_mask[i, :L] = item["attention_mask"]
        n = int(item["sent_len"])
        word_first_wp[i, :n]  = item["word_first_wp_fullidx"]
        sent_lens[i]           = n
        sentence_mask[i, :n]  = True
        if has_emotion:
            emotion_labels[i] = item["emotion_labels"]
        for f, fr in enumerate(item.get("frames", [])):
            frames_mask[i, f]          = True
            frames_labels[i, f, :n]    = fr["labels"]
            frames_pred_word_idx[i, f] = fr["pred_word_idx"]
            frames_role_ids[i, f, :n]  = fr["role_ids"]
            frames_arg0_mask[i, f, :n] = fr["arg0_mask"]
            frames_arg1_mask[i, f, :n] = fr["arg1_mask"]
            frames_arg2_mask[i, f, :n] = fr["arg2_mask"]
            frames_argm_mask[i, f, :n] = fr["argm_mask"]

    res = {
        "input_ids": input_ids, "token_type_ids": token_type_ids, "attention_mask": attention_mask,
        "word_first_wp_fullidx": word_first_wp, "sentence_mask": sentence_mask, "sent_lens": sent_lens,
        "frames_mask": frames_mask, "frames_labels": frames_labels,
        "frames_pred_word_idx": frames_pred_word_idx, "frames_role_ids": frames_role_ids,
        "frames_arg0_mask": frames_arg0_mask, "frames_arg1_mask": frames_arg1_mask,
        "frames_arg2_mask": frames_arg2_mask, "frames_argm_mask": frames_argm_mask,
    }
    if has_emotion:
        res["emotion_labels"] = emotion_labels
    return res


# ---- Vanilla (no SRL) ----
class UtteranceOnlyDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=128):
        self.samples    = samples
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        utt = self.samples[idx]
        enc = self.tokenizer(" ".join(utt.words), truncation=True,
                             max_length=self.max_length, padding=False, return_tensors=None)
        res = {
            "input_ids":      torch.tensor(enc["input_ids"],      dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
        }
        if "token_type_ids" in enc:
            res["token_type_ids"] = torch.tensor(enc["token_type_ids"], dtype=torch.long)
        if utt.emotions is not None:
            res["emotion_labels"] = emotions_to_vector(utt.emotions, num_emotions=NUM_EMOTIONS)
        return res


def utterance_collate(batch, pad_token_id):
    B     = len(batch)
    max_L = max(x["input_ids"].size(0) for x in batch)
    input_ids      = torch.full((B, max_L), pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((B, max_L), dtype=torch.long)
    has_ttids  = "token_type_ids" in batch[0]
    has_emo    = "emotion_labels" in batch[0]
    if has_ttids: token_type_ids = torch.zeros((B, max_L), dtype=torch.long)
    if has_emo:   emotion_labels = torch.zeros((B, NUM_EMOTIONS), dtype=torch.float)

    for i, item in enumerate(batch):
        L = item["input_ids"].size(0)
        input_ids[i, :L]      = item["input_ids"]
        attention_mask[i, :L] = item["attention_mask"]
        if has_ttids: token_type_ids[i, :L] = item["token_type_ids"]
        if has_emo:   emotion_labels[i]      = item["emotion_labels"]

    res = {"input_ids": input_ids, "attention_mask": attention_mask}
    if has_ttids: res["token_type_ids"] = token_type_ids
    if has_emo:   res["emotion_labels"] = emotion_labels
    return res

# 3. Models for Role-attention (PRED + ARG1 + ARGM)

### vanila mlp/nomlp

In [ ]:
# ============================================================
#  Vanilla BERT 
# ============================================================
class VanillaEmotionClassifier_mlp(nn.Module):
    def __init__(self, bert_name: str, num_emotions=None, mlp_hidden=256,dropout=0.1):
        super().__init__()
        if num_emotions is None:
            num_emotions = NUM_EMOTIONS
        
        self.config = AutoConfig.from_pretrained(bert_name)
        self.bert   = AutoModel.from_pretrained(bert_name)
        # H = self.config.hidden_size
        H = 768
        self.dropout  = nn.Dropout(dropout)
        self.cls_head = nn.Sequential(
            nn.Linear(H, mlp_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, num_emotions)
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None, emotion_labels=None, **kwargs):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled = out.pooler_output if (hasattr(out, "pooler_output") and out.pooler_output is not None) \
                 else out.last_hidden_state[:, 0, :]
        logits = self.cls_head(self.dropout(pooled))        # [B, num_emotions]
        loss   = nn.BCEWithLogitsLoss()(logits, emotion_labels) if emotion_labels is not None \
                 else torch.tensor(0.0, device=logits.device)
        return logits, loss, None

#no mlp
class VanillaEmotionClassifier(nn.Module):
    def __init__(self, bert_name: str, num_emotions=None, dropout=0.1):
        super().__init__()
        if num_emotions is None:
            num_emotions = NUM_EMOTIONS

        self.config = AutoConfig.from_pretrained(bert_name)
        self.bert = AutoModel.from_pretrained(bert_name)
        H = self.config.hidden_size   # 768 for bert-base

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(H, num_emotions)

    def forward(self, input_ids, attention_mask, token_type_ids=None, emotion_labels=None, **kwargs):
        out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        pooled = out.pooler_output if (hasattr(out, "pooler_output") and out.pooler_output is not None) \
                 else out.last_hidden_state[:, 0, :]

        logits = self.classifier(self.dropout(pooled))
        loss = nn.BCEWithLogitsLoss()(logits, emotion_labels) if emotion_labels is not None \
               else torch.tensor(0.0, device=logits.device)
        return logits, loss, None

### nofusion-nomlp

In [ ]:
class SRLEmotionClassifierAttention_nofusion_three_nomlp(nn.Module):
    """
    SRL-aware emotion classifier with role-aware attention/gating (Pred, ARG1, ARGM).
    - Keeps vanilla CLS pooling
    - Builds role vectors per frame
    - Uses CLS-conditioned attention over roles (mask-aware)
    - Averages over frames (mask-aware)
    - Uses a GoEmotions-style top head: Dropout -> Linear(head_in, num_emotions)
    """

    def __init__(
        self,
        bert_name: str,
        num_labels: int,
        num_emotions=NUM_EMOTIONS,
        dropout=0.1,
        role_feat_dim=256,
        use_role_attention: bool = True,
        attn_dim: int = 256,
    ):
        super().__init__()
        self.config = AutoConfig.from_pretrained(bert_name)
        self.bert = AutoModel.from_pretrained(bert_name)
        H = self.config.hidden_size

        self.attn_dim = attn_dim
        self.dropout = nn.Dropout(dropout)
        self.use_role_attention = use_role_attention

        # --- role-aware attention modules (3 roles: Pred, ARG1, ARGM) ---
        if self.use_role_attention:
            self.q_proj = nn.Linear(H, attn_dim)   # CLS -> query
            self.k_proj = nn.Linear(H, attn_dim)   # role vec -> key
            self.v_proj = nn.Linear(H, attn_dim)   # role vec -> value
            self.role_out = nn.Sequential(
                nn.Linear(attn_dim, role_feat_dim),
                nn.LayerNorm(role_feat_dim),
                nn.Tanh()
            )
            head_in = H + role_feat_dim
        else:
            head_in = H

        # GoEmotions-style classification head:
        # pooled/combined representation -> dropout -> linear to labels
        self.cls_head = nn.Linear(head_in, num_emotions)

        self._bce = nn.BCEWithLogitsLoss()

    @staticmethod
    def _masked_mean(X: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
        """
        X: [BF, N, H]
        m: [BF, N] float/bool mask
        -> [BF, H]
        """
        m = m.float()
        den = m.sum(1, keepdim=True).clamp(min=1.0)
        return (X * m.unsqueeze(-1)).sum(1) / den

    def forward(
        self,
        input_ids,
        token_type_ids=None,
        attention_mask=None,
        word_first_wp_fullidx=None,
        sentence_mask=None,
        sent_lens=None,
        frames_pred_word_idx=None,
        frames_role_ids=None,
        frames_arg0_mask=None,
        frames_arg1_mask=None,
        frames_arg2_mask=None,
        frames_argm_mask=None,
        frames_mask=None,
        emotion_labels=None,
        **kwargs
    ):
        device = input_ids.device

        out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        Htok = out.last_hidden_state  # [B, T, H]

        pooled = (
            out.pooler_output
            if (hasattr(out, "pooler_output") and out.pooler_output is not None)
            else Htok[:, 0, :]
        )  # [B, H]

        role_ctx = None

        can_use_srl = (
            self.use_role_attention
            and (sentence_mask is not None)
            and (word_first_wp_fullidx is not None)
            and (frames_pred_word_idx is not None)
            and (frames_mask is not None)
        )

        if can_use_srl:
            B = input_ids.size(0)
            N = sentence_mask.size(1)
            F = frames_mask.size(1)
            BF = B * F

            # word-level representations aligned to N words: [B, N, H]
            g = word_first_wp_fullidx.clone()
            g[g < 0] = 0
            g = g.unsqueeze(-1).expand(-1, -1, Htok.size(-1))
            H_w = torch.gather(Htok, 1, g) * sentence_mask.unsqueeze(-1).float()

            # expand per frame
            Hf = (
                H_w.unsqueeze(1)
                .expand(B, F, N, H_w.size(-1))
                .contiguous()
                .view(BF, N, -1)
            )
            smf = (
                sentence_mask.unsqueeze(1)
                .expand(B, F, N)
                .contiguous()
                .view(BF, N)
                .float()
            )
            fm = frames_mask.float()  # [B, F]

            # predicate mask per frame: [BF, N]
            pred_idx = frames_pred_word_idx.clamp(min=0, max=N - 1).contiguous().view(BF)
            pred_mask = torch.zeros(BF, N, device=device, dtype=smf.dtype)
            pred_mask.scatter_(1, pred_idx.unsqueeze(1), 1.0)
            pred_mask = pred_mask * smf

            # role masks: [BF, N]
            def _role_mask(m):
                if m is None:
                    return torch.zeros(BF, N, device=device, dtype=smf.dtype)
                return m.contiguous().view(BF, N).float() * smf

            a1 = _role_mask(frames_arg1_mask)
            am = _role_mask(frames_argm_mask)

            # pooled role vectors: [BF, H]
            v_pred = self._masked_mean(Hf, pred_mask)
            v_a1 = self._masked_mean(Hf, a1)
            v_am = self._masked_mean(Hf, am)

            # role existence mask: [BF, 3]
            pred_exists = (pred_mask.sum(-1) > 0).float()
            a1_exists = (a1.sum(-1) > 0).float()
            am_exists = (am.sum(-1) > 0).float()
            role_exists = torch.stack([pred_exists, a1_exists, am_exists], dim=-1)

            # stack role vectors: [BF, 3, H]
            V_roles = torch.stack([v_pred, v_a1, v_am], dim=1)

            # CLS query per frame: [BF, attn_dim]
            q = (
                self.q_proj(pooled)
                .unsqueeze(1)
                .expand(B, F, -1)
                .contiguous()
                .view(BF, -1)
            )

            # keys/values: [BF, 3, attn_dim]
            k = self.k_proj(V_roles)
            v = self.v_proj(V_roles)

            # attention scores: [BF, 3]
            scores = (k * q.unsqueeze(1)).sum(-1) / (k.size(-1) ** 0.5)

            # mask missing roles
            scores = scores.masked_fill(role_exists <= 0, float("-inf"))

            # avoid NaNs if all roles are missing
            all_inf = torch.isinf(scores).all(dim=-1, keepdim=True)
            scores = torch.where(all_inf, torch.zeros_like(scores), scores)

            alpha = torch.softmax(scores, dim=-1)  # [BF, 3]

            # context per frame: [BF, attn_dim]
            ctx_f = (alpha.unsqueeze(-1) * v).sum(1)

            # project -> [BF, role_feat_dim]
            role_feat = self.role_out(ctx_f)

            # average across frames -> [B, role_feat_dim]
            role_feat = role_feat.view(B, F, -1)
            role_ctx = (
                (role_feat * fm.unsqueeze(-1)).sum(1)
                / fm.sum(1).clamp(min=1.0).unsqueeze(-1)
            )

        # final representation
        if role_ctx is not None:
            rep = torch.cat([self.dropout(pooled), self.dropout(role_ctx)], dim=-1)
        else:
            rep = self.dropout(pooled)

        logits = self.cls_head(rep)  # [B, E]

        loss = (
            self._bce(logits, emotion_labels)
            if emotion_labels is not None
            else torch.tensor(0.0, device=logits.device)
        )

        return logits, loss, None

### nofusion-mlp

In [ ]:
class SRLEmotionClassifierAttention_nofusion_three_mlp(nn.Module):
    """
    SRL-aware emotion classifier with role-aware attention over three roles
    (Pred, ARG1, ARGM), no gated fusion, and an MLP classification head.
    """

    def __init__(
        self,
        bert_name: str,
        num_labels: int,
        num_emotions=NUM_EMOTIONS,
        mlp_hidden=256,
        dropout=0.1,
        role_feat_dim=256,
        use_role_attention: bool = True,
        attn_dim: int = 256,
    ):
        super().__init__()
        self.config = AutoConfig.from_pretrained(bert_name)
        self.bert = AutoModel.from_pretrained(bert_name)
        H = self.config.hidden_size
        
        self.attn_dim = attn_dim
        self.dropout = nn.Dropout(dropout)
        self.use_role_attention = use_role_attention

        # --- role-aware attention modules (3 roles: Pred, ARG1, ARGM) ---
        if self.use_role_attention:
            self.q_proj = nn.Linear(H, attn_dim)   # CLS -> query
            self.k_proj = nn.Linear(H, attn_dim)   # role vec -> key
            self.v_proj = nn.Linear(H, attn_dim)   # role vec -> value
            self.role_out = nn.Sequential(
                nn.Linear(attn_dim, role_feat_dim),
                nn.LayerNorm(role_feat_dim),
                nn.Tanh()
            )
            head_in = H + role_feat_dim
        else:
            head_in = H

        # CHANGED: MLP head added
        self.cls_head = nn.Sequential(
            nn.Linear(head_in, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, num_emotions)
        )

        self._bce = nn.BCEWithLogitsLoss()

    @staticmethod
    def _masked_mean(X: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
        """
        X: [BF, N, H]
        m: [BF, N] float/bool mask
        -> [BF, H]
        """
        m = m.float()
        den = m.sum(1, keepdim=True).clamp(min=1.0)
        return (X * m.unsqueeze(-1)).sum(1) / den

    def forward(
        self,
        input_ids,
        token_type_ids=None,
        attention_mask=None,
        word_first_wp_fullidx=None,
        sentence_mask=None,
        sent_lens=None,
        frames_pred_word_idx=None,
        frames_role_ids=None,
        frames_arg0_mask=None,
        frames_arg1_mask=None,
        frames_arg2_mask=None,
        frames_argm_mask=None,
        frames_mask=None,
        emotion_labels=None,
        **kwargs
    ):
        device = input_ids.device

        out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        Htok = out.last_hidden_state  # [B, T, H]

        pooled = (
            out.pooler_output
            if (hasattr(out, "pooler_output") and out.pooler_output is not None)
            else Htok[:, 0, :]
        )  # [B, H]

        role_ctx = None

        can_use_srl = (
            self.use_role_attention
            and (sentence_mask is not None)
            and (word_first_wp_fullidx is not None)
            and (frames_pred_word_idx is not None)
            and (frames_mask is not None)
        )

        if can_use_srl:
            B = input_ids.size(0)
            N = sentence_mask.size(1)
            F = frames_mask.size(1)

            # map word-level reps
            gather_idx = word_first_wp_fullidx.clone()
            gather_idx[gather_idx < 0] = 0
            gather_idx = gather_idx.unsqueeze(-1).expand(-1, -1, Htok.size(-1))
            H_words = torch.gather(Htok, dim=1, index=gather_idx)
            H_words = H_words * sentence_mask.unsqueeze(-1)

            # expand across frames
            BF = B * F
            Hf = H_words.unsqueeze(1).expand(B, F, N, H_words.size(-1)).contiguous()
            Hf = Hf.view(BF, N, H_words.size(-1))

            sent_mask_f = sentence_mask.unsqueeze(1).expand(B, F, N).contiguous().view(BF, N)
            pred_idx_f = frames_pred_word_idx.contiguous().view(BF)

            # predicate mask
            pred_mask = torch.zeros((BF, N), device=device)
            pred_mask.scatter_(1, pred_idx_f.unsqueeze(1).clamp(min=0, max=N - 1), 1.0)
            pred_mask = pred_mask * sent_mask_f.float()

            # ARG1 / ARGM masks
            arg1_f = frames_arg1_mask.contiguous().view(BF, N).float() if frames_arg1_mask is not None else torch.zeros((BF, N), device=device)
            argm_f = frames_argm_mask.contiguous().view(BF, N).float() if frames_argm_mask is not None else torch.zeros((BF, N), device=device)

            arg1_f = arg1_f * sent_mask_f.float()
            argm_f = argm_f * sent_mask_f.float()

            # pooled role vectors
            v_pred = self._masked_mean(Hf, pred_mask)
            v_arg1 = self._masked_mean(Hf, arg1_f)
            v_argm = self._masked_mean(Hf, argm_f)

            V_roles = torch.stack([v_pred, v_arg1, v_argm], dim=1)  # [BF, 3, H]

            # CLS query repeated per frame
            q = self.q_proj(pooled).unsqueeze(1).expand(B, F, -1).contiguous().view(BF, -1)
            k = self.k_proj(V_roles)
            v = self.v_proj(V_roles)

            # role existence mask
            role_exists = torch.stack([
                (pred_mask.sum(dim=1) > 0),
                (arg1_f.sum(dim=1) > 0),
                (argm_f.sum(dim=1) > 0),
            ], dim=1)

            scores = (k * q.unsqueeze(1)).sum(-1) / (self.attn_dim ** 0.5)
            scores = scores.masked_fill(~role_exists, -10000.0)

            alpha = torch.softmax(scores, dim=1)
            ctx = (alpha.unsqueeze(-1) * v).sum(dim=1)  # [BF, attn_dim]
            role_feat = self.role_out(ctx)              # [BF, role_feat_dim]

            # aggregate across valid frames
            fm = frames_mask.float()
            role_feat = role_feat.view(B, F, -1)
            role_ctx = (role_feat * fm.unsqueeze(-1)).sum(dim=1) / fm.sum(dim=1, keepdim=True).clamp(min=1.0)

        # no fusion: just concat pooled + role_ctx
        if role_ctx is not None:
            rep = torch.cat([self.dropout(pooled), self.dropout(role_ctx)], dim=-1)
        else:
            rep = self.dropout(pooled)

        logits = self.cls_head(rep)

        loss = torch.tensor(0.0, device=logits.device)
        if emotion_labels is not None:
            loss = self._bce(logits, emotion_labels)

        return logits, loss, None

### fusion_nomlp

In [ ]:
#Gated-fusuion version

class SRLEmotionClassifierAttention_fusion_three_nomlp(nn.Module):
    """
    SRL-aware emotion classifier with role-aware attention/gating (Pred, ARG1, ARGM).
    - Keeps vanilla CLS pooling
    - Builds role vectors per frame
    - Uses CLS-conditioned attention over roles (mask-aware)
    - Averages over frames (mask-aware)
    - Uses gated residual fusion between pooled text representation and SRL role representation
    """

    def __init__(
        self,
        bert_name: str,
        num_labels: int,
        num_emotions=NUM_EMOTIONS,
        dropout=0.1,
        role_feat_dim=256,
        use_role_attention: bool = True,
        attn_dim: int = 256,
    ):
        super().__init__()
        self.config = AutoConfig.from_pretrained(bert_name)
        self.bert = AutoModel.from_pretrained(bert_name)
        H = self.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.use_role_attention = use_role_attention
        self.hidden_size = H

        # --- role-aware attention modules (3 roles: Pred, ARG1, ARGM) ---
        if self.use_role_attention:
            self.q_proj = nn.Linear(H, attn_dim)   # CLS -> query
            self.k_proj = nn.Linear(H, attn_dim)   # role vec -> key
            self.v_proj = nn.Linear(H, attn_dim)   # role vec -> value
            self.role_out = nn.Sequential(
                nn.Linear(attn_dim, role_feat_dim),
                nn.LayerNorm(role_feat_dim),
                nn.Tanh()
            )

            # =========================
            # # NEW: fusion components
            # =========================
            # # Project SRL role vector into same hidden space as pooled BERT
            self.role_to_hidden = nn.Linear(role_feat_dim, H)

            # # Learn gate from [pooled ; role_ctx]
            self.gate_proj = nn.Linear(H + role_feat_dim, H)

            # # NEW: classifier now takes fused H-sized vector, not concat(H + role_feat_dim)
            self.cls_head = nn.Linear(H, num_emotions)

        else:
            # # If SRL is disabled, fall back to vanilla pooled output
            self.cls_head = nn.Linear(H, num_emotions)

        self._bce = nn.BCEWithLogitsLoss()

    @staticmethod
    def _masked_mean(X: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
        """
        X: [BF, N, H]
        m: [BF, N] float/bool mask
        -> [BF, H]
        """
        m = m.float()
        den = m.sum(1, keepdim=True).clamp(min=1.0)
        return (X * m.unsqueeze(-1)).sum(1) / den

    def forward(
        self,
        input_ids,
        token_type_ids=None,
        attention_mask=None,
        word_first_wp_fullidx=None,
        sentence_mask=None,
        sent_lens=None,
        frames_pred_word_idx=None,
        frames_role_ids=None,
        frames_arg0_mask=None,
        frames_arg1_mask=None,
        frames_arg2_mask=None,
        frames_argm_mask=None,
        frames_mask=None,
        emotion_labels=None,
        **kwargs
    ):
        device = input_ids.device

        out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        Htok = out.last_hidden_state  # [B, T, H]

        pooled = (
            out.pooler_output
            if (hasattr(out, "pooler_output") and out.pooler_output is not None)
            else Htok[:, 0, :]
        )  # [B, H]

        role_ctx = None

        can_use_srl = (
            self.use_role_attention
            and (sentence_mask is not None)
            and (word_first_wp_fullidx is not None)
            and (frames_pred_word_idx is not None)
            and (frames_mask is not None)
        )

        if can_use_srl:
            B = input_ids.size(0)
            N = sentence_mask.size(1)
            F = frames_mask.size(1)
            BF = B * F

            # word-level representations aligned to N words: [B, N, H]
            g = word_first_wp_fullidx.clone()
            g[g < 0] = 0
            g = g.unsqueeze(-1).expand(-1, -1, Htok.size(-1))
            H_w = torch.gather(Htok, 1, g) * sentence_mask.unsqueeze(-1).float()

            # expand per frame
            Hf = (
                H_w.unsqueeze(1)
                .expand(B, F, N, H_w.size(-1))
                .contiguous()
                .view(BF, N, -1)
            )
            smf = (
                sentence_mask.unsqueeze(1)
                .expand(B, F, N)
                .contiguous()
                .view(BF, N)
                .float()
            )
            fm = frames_mask.float()  # [B, F]

            # predicate mask per frame: [BF, N]
            pred_idx = frames_pred_word_idx.clamp(min=0, max=N - 1).contiguous().view(BF)
            pred_mask = torch.zeros(BF, N, device=device, dtype=smf.dtype)
            pred_mask.scatter_(1, pred_idx.unsqueeze(1), 1.0)
            pred_mask = pred_mask * smf

            # role masks: [BF, N]
            def _role_mask(m):
                if m is None:
                    return torch.zeros(BF, N, device=device, dtype=smf.dtype)
                return m.contiguous().view(BF, N).float() * smf

            a1 = _role_mask(frames_arg1_mask)
            am = _role_mask(frames_argm_mask)

            # pooled role vectors: [BF, H]
            v_pred = self._masked_mean(Hf, pred_mask)
            v_a1 = self._masked_mean(Hf, a1)
            v_am = self._masked_mean(Hf, am)

            # role existence mask: [BF, 3]
            pred_exists = (pred_mask.sum(-1) > 0).float()
            a1_exists = (a1.sum(-1) > 0).float()
            am_exists = (am.sum(-1) > 0).float()
            role_exists = torch.stack([pred_exists, a1_exists, am_exists], dim=-1)

            # stack role vectors: [BF, 3, H]
            V_roles = torch.stack([v_pred, v_a1, v_am], dim=1)

            # CLS query per frame: [BF, attn_dim]
            q = (
                self.q_proj(pooled)
                .unsqueeze(1)
                .expand(B, F, -1)
                .contiguous()
                .view(BF, -1)
            )

            # keys/values: [BF, 3, attn_dim]
            k = self.k_proj(V_roles)
            v = self.v_proj(V_roles)

            # attention scores: [BF, 3]
            scores = (k * q.unsqueeze(1)).sum(-1) / (k.size(-1) ** 0.5)

            # mask missing roles
            scores = scores.masked_fill(role_exists <= 0, float("-inf"))

            # avoid NaNs if all roles are missing
            all_inf = torch.isinf(scores).all(dim=-1, keepdim=True)
            scores = torch.where(all_inf, torch.zeros_like(scores), scores)

            alpha = torch.softmax(scores, dim=-1)  # [BF, 3]

            # context per frame: [BF, attn_dim]
            ctx_f = (alpha.unsqueeze(-1) * v).sum(1)

            # project -> [BF, role_feat_dim]
            role_feat = self.role_out(ctx_f)

            # average across frames -> [B, role_feat_dim]
            role_feat = role_feat.view(B, F, -1)
            role_ctx = (
                (role_feat * fm.unsqueeze(-1)).sum(1)
                / fm.sum(1).clamp(min=1.0).unsqueeze(-1)
            )

        # ============================
        # # NEW: gated residual fusion
        # ============================
        if role_ctx is not None:
            # # Project SRL role representation to BERT hidden space
            role_h = self.role_to_hidden(self.dropout(role_ctx))   # [B, H]

            # # Build gate from text + role features
            gate_in = torch.cat([pooled, role_ctx], dim=-1)        # [B, H + role_feat_dim]
            gate = torch.sigmoid(self.gate_proj(self.dropout(gate_in)))  # [B, H]

            # # Residual fusion: role branch selectively updates pooled text rep
            fused = pooled + gate * role_h                         # [B, H]

            rep = self.dropout(fused)
        else:
            # # Fallback to vanilla pooled representation
            rep = self.dropout(pooled)

        logits = self.cls_head(rep)  # [B, E]

        loss = (
            self._bce(logits, emotion_labels)
            if emotion_labels is not None
            else torch.tensor(0.0, device=logits.device)
        )

        return logits, loss, None

### fusion -mlp

In [ ]:
# Gated-fusion version + MLP head

class SRLEmotionClassifierAttention_fusion_three_mlp(nn.Module):
    """
    SRL-aware emotion classifier with role-aware attention/gating (Pred, ARG1, ARGM).
    - Keeps vanilla CLS pooling
    - Builds role vectors per frame
    - Uses CLS-conditioned attention over roles (mask-aware)
    - Averages over frames (mask-aware)
    - Uses gated residual fusion between pooled text representation and SRL role representation
    - Uses an MLP classification head
    """

    def __init__(
        self,
        bert_name: str,
        num_labels: int,
        num_emotions=NUM_EMOTIONS,
        mlp_hidden=256,
        dropout=0.1,
        role_feat_dim=256,
        use_role_attention: bool = True,
        attn_dim: int = 256,
    ):
        super().__init__()
        self.config = AutoConfig.from_pretrained(bert_name)
        self.bert = AutoModel.from_pretrained(bert_name)
        H = self.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.use_role_attention = use_role_attention
        self.hidden_size = H

        # --- role-aware attention modules (3 roles: Pred, ARG1, ARGM) ---
        if self.use_role_attention:
            self.q_proj = nn.Linear(H, attn_dim)   # CLS -> query
            self.k_proj = nn.Linear(H, attn_dim)   # role vec -> key
            self.v_proj = nn.Linear(H, attn_dim)   # role vec -> value
            self.role_out = nn.Sequential(
                nn.Linear(attn_dim, role_feat_dim),
                nn.LayerNorm(role_feat_dim),
                nn.Tanh()
            )

            # fusion components
            self.role_to_hidden = nn.Linear(role_feat_dim, H)
            self.gate_proj = nn.Linear(H + role_feat_dim, H)

            # CHANGED: MLP head instead of single linear
            self.cls_head = nn.Sequential(
                nn.Linear(H, mlp_hidden),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(mlp_hidden, num_emotions)
            )

        else:
            # fallback if SRL is disabled
            self.cls_head = nn.Sequential(
                nn.Linear(H, mlp_hidden),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(mlp_hidden, num_emotions)
            )

        self._bce = nn.BCEWithLogitsLoss()

    @staticmethod
    def _masked_mean(X: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
        """
        X: [BF, N, H]
        m: [BF, N] float/bool mask
        -> [BF, H]
        """
        m = m.float()
        den = m.sum(1, keepdim=True).clamp(min=1.0)
        return (X * m.unsqueeze(-1)).sum(1) / den

    def forward(
        self,
        input_ids,
        token_type_ids=None,
        attention_mask=None,
        word_first_wp_fullidx=None,
        sentence_mask=None,
        sent_lens=None,
        frames_pred_word_idx=None,
        frames_role_ids=None,
        frames_arg0_mask=None,
        frames_arg1_mask=None,
        frames_arg2_mask=None,
        frames_argm_mask=None,
        frames_mask=None,
        emotion_labels=None,
        **kwargs
    ):
        device = input_ids.device

        out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        Htok = out.last_hidden_state  # [B, T, H]

        pooled = (
            out.pooler_output
            if (hasattr(out, "pooler_output") and out.pooler_output is not None)
            else Htok[:, 0, :]
        )  # [B, H]

        role_ctx = None

        can_use_srl = (
            self.use_role_attention
            and (sentence_mask is not None)
            and (word_first_wp_fullidx is not None)
            and (frames_pred_word_idx is not None)
            and (frames_mask is not None)
        )

        if can_use_srl:
            B = input_ids.size(0)
            N = sentence_mask.size(1)
            F = frames_mask.size(1)
            BF = B * F

            # word-level representations aligned to N words: [B, N, H]
            g = word_first_wp_fullidx.clone()
            g[g < 0] = 0
            g = g.unsqueeze(-1).expand(-1, -1, Htok.size(-1))
            H_w = torch.gather(Htok, 1, g) * sentence_mask.unsqueeze(-1).float()

            # expand per frame
            Hf = (
                H_w.unsqueeze(1)
                .expand(B, F, N, H_w.size(-1))
                .contiguous()
                .view(BF, N, -1)
            )
            smf = (
                sentence_mask.unsqueeze(1)
                .expand(B, F, N)
                .contiguous()
                .view(BF, N)
                .float()
            )
            fm = frames_mask.float()  # [B, F]

            # predicate mask per frame: [BF, N]
            pred_idx = frames_pred_word_idx.clamp(min=0, max=N - 1).contiguous().view(BF)
            pred_mask = torch.zeros(BF, N, device=device, dtype=smf.dtype)
            pred_mask.scatter_(1, pred_idx.unsqueeze(1), 1.0)
            pred_mask = pred_mask * smf

            # role masks: [BF, N]
            def _role_mask(m):
                if m is None:
                    return torch.zeros(BF, N, device=device, dtype=smf.dtype)
                return m.contiguous().view(BF, N).float() * smf

            a1 = _role_mask(frames_arg1_mask)
            am = _role_mask(frames_argm_mask)

            # pooled role vectors: [BF, H]
            v_pred = self._masked_mean(Hf, pred_mask)
            v_a1 = self._masked_mean(Hf, a1)
            v_am = self._masked_mean(Hf, am)

            # role existence mask: [BF, 3]
            pred_exists = (pred_mask.sum(-1) > 0).float()
            a1_exists = (a1.sum(-1) > 0).float()
            am_exists = (am.sum(-1) > 0).float()
            role_exists = torch.stack([pred_exists, a1_exists, am_exists], dim=-1)

            # stack role vectors: [BF, 3, H]
            V_roles = torch.stack([v_pred, v_a1, v_am], dim=1)

            # CLS query per frame: [BF, attn_dim]
            q = (
                self.q_proj(pooled)
                .unsqueeze(1)
                .expand(B, F, -1)
                .contiguous()
                .view(BF, -1)
            )

            # keys/values: [BF, 3, attn_dim]
            k = self.k_proj(V_roles)
            v = self.v_proj(V_roles)

            # attention scores: [BF, 3]
            scores = (k * q.unsqueeze(1)).sum(-1) / (k.size(-1) ** 0.5)

            # mask missing roles
            scores = scores.masked_fill(role_exists <= 0, float("-inf"))

            # avoid NaNs if all roles are missing
            all_inf = torch.isinf(scores).all(dim=-1, keepdim=True)
            scores = torch.where(all_inf, torch.zeros_like(scores), scores)

            alpha = torch.softmax(scores, dim=-1)  # [BF, 3]

            # context per frame: [BF, attn_dim]
            ctx_f = (alpha.unsqueeze(-1) * v).sum(1)

            # project -> [BF, role_feat_dim]
            role_feat = self.role_out(ctx_f)

            # average across frames -> [B, role_feat_dim]
            role_feat = role_feat.view(B, F, -1)
            role_ctx = (
                (role_feat * fm.unsqueeze(-1)).sum(1)
                / fm.sum(1).clamp(min=1.0).unsqueeze(-1)
            )

        # gated residual fusion
        if role_ctx is not None:
            role_h = self.role_to_hidden(self.dropout(role_ctx))   # [B, H]
            gate_in = torch.cat([pooled, role_ctx], dim=-1)        # [B, H + role_feat_dim]
            gate = torch.sigmoid(self.gate_proj(self.dropout(gate_in)))  # [B, H]
            fused = pooled + gate * role_h                         # [B, H]
            rep = self.dropout(fused)
        else:
            rep = self.dropout(pooled)

        logits = self.cls_head(rep)  # [B, E]

        loss = (
            self._bce(logits, emotion_labels)
            if emotion_labels is not None
            else torch.tensor(0.0, device=logits.device)
        )

        return logits, loss, None

# 4. Training & Evaluation

In [ ]:
# CHANGED: helper for partial checkpoint transfer
def load_partial_emotion_checkpoint(model, ckpt_path, old_emo_list, new_emo_list, device):
    """
    Load:
      1) bert.* weights
      2) cls_head.0 fully if shape matches
      3) cls_head.3 row-by-row for shared emotions only

    Old head can include extra emotions (e.g., neutral).
    New unmatched emotions stay randomly initialized.
    """
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    state_dict = ckpt["model_state"] if "model_state" in ckpt else ckpt

    # 1) load bert encoder
    bert_state = {k: v for k, v in state_dict.items() if k.startswith("bert.")}
    missing, unexpected = model.load_state_dict(bert_state, strict=False)
    print("After loading bert:")
    print("  Missing keys:", missing)
    print("  Unexpected keys:", unexpected)

    with torch.no_grad():
        # 2) load first MLP layer if compatible
        if "cls_head.0.weight" in state_dict and model.cls_head[0].weight.shape == state_dict["cls_head.0.weight"].shape:
            model.cls_head[0].weight.copy_(state_dict["cls_head.0.weight"])
            print("Copied cls_head.0.weight")
        else:
            print("Skipped cls_head.0.weight")

        if "cls_head.0.bias" in state_dict and model.cls_head[0].bias.shape == state_dict["cls_head.0.bias"].shape:
            model.cls_head[0].bias.copy_(state_dict["cls_head.0.bias"])
            print("Copied cls_head.0.bias")
        else:
            print("Skipped cls_head.0.bias")

        # 3) load final output layer row-by-row for shared emotions
        if "cls_head.3.weight" not in state_dict or "cls_head.3.bias" not in state_dict:
            print("Skipped cls_head.3.* because keys do not exist in checkpoint")
            return model

        old_out_w = state_dict["cls_head.3.weight"]   # [old_num_emotions, mlp_hidden]
        old_out_b = state_dict["cls_head.3.bias"]     # [old_num_emotions]

        old_idx = {emo: i for i, emo in enumerate(old_emo_list)}
        new_idx = {emo: i for i, emo in enumerate(new_emo_list)}

        copied = []
        skipped = []

        for emo in new_emo_list:
            if emo in old_idx:
                oi = old_idx[emo]
                ni = new_idx[emo]

                # CHANGED: extra shape safety
                if oi < old_out_w.size(0) and ni < model.cls_head[3].weight.size(0):
                    model.cls_head[3].weight[ni].copy_(old_out_w[oi])
                    model.cls_head[3].bias[ni].copy_(old_out_b[oi])
                    copied.append(emo)
                else:
                    skipped.append(emo)
            else:
                skipped.append(emo)

        print("Copied output rows for:", copied)
        print("Randomly initialized rows for:", skipped)

    return model


def train_one_epoch(model, dataloader, optimizer, device, scheduler=None,
                    grad_accum_steps=1, amp=True, max_grad_norm=1.0):
    model.train()
    total_loss, n = 0.0, 0
    use_amp = amp and torch.cuda.is_available()
    scaler  = torch.amp.GradScaler("cuda", enabled=use_amp)
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(dataloader, 1):
        batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
        with torch.amp.autocast("cuda", enabled=use_amp, dtype=torch.float16):
            _, loss, _ = model(**batch)
        total_loss += float(loss.detach())
        n += 1
        scaled = loss / grad_accum_steps
        if use_amp: scaler.scale(scaled).backward()
        else:       scaled.backward()
        if step % grad_accum_steps == 0:
            if use_amp: scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            if use_amp: scaler.step(optimizer); scaler.update()
            else:       optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            if scheduler: scheduler.step()
    return total_loss / max(1, n)


# CHANGED: classification threshold tuning is no longer needed for regression
# def tune_per_class_thresholds(...):
#     ...


@torch.no_grad()
def collect_preds_and_golds(model, dataloader, device):   # CHANGED: function name
    model.eval()
    all_preds, all_golds = [], []   # CHANGED
    
    for batch in dataloader:
        batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
        preds, _, _ = model(**batch)   # CHANGED: model already returns 1~5 predictions

        preds = preds.detach().cpu()   # CHANGED
        golds = batch["emotion_labels"].detach().cpu()

        all_preds.append(preds)   # CHANGED
        all_golds.append(golds)
        
    if not all_preds:   # CHANGED
        return np.array([]), np.array([])

    all_preds = torch.cat(all_preds).numpy()   # CHANGED
    all_golds = torch.cat(all_golds).numpy()
    return all_preds, all_golds

#This is validation
@torch.no_grad()
def eval_emotion(model, dataloader, device, emo_list=None):
    model.eval()
    total_loss, n = 0.0, 0
    all_preds, all_golds = [], []

    for batch in dataloader:
        batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
        preds, loss, _ = model(**batch)
        total_loss += float(loss)
        n += 1

        if "emotion_labels" in batch:
            all_preds.append(preds.cpu())
            all_golds.append(batch["emotion_labels"].cpu())

    if not all_preds:
        return total_loss / max(1, n), 0.0, 0.0

    all_preds = torch.cat(all_preds).numpy()
    all_golds = torch.cat(all_golds).numpy()

    # overall flattened correlation
    pearson_all = pearsonr(all_golds.ravel(), all_preds.ravel())[0]
    spearman_all = spearmanr(all_golds.ravel(), all_preds.ravel())[0]

    print(f"Overall Pearson:  {pearson_all:.4f}")
    print(f"Overall Spearman: {spearman_all:.4f}")

    # per-emotion correlation
    per_emotion_pearson = []
    per_emotion_spearman = []

    for i in range(all_golds.shape[1]):
        g = all_golds[:, i]
        p = all_preds[:, i]

        # avoid crash if one side is constant
        if np.std(g) == 0 or np.std(p) == 0:
            pr = np.nan
            sr = np.nan
        else:
            pr = pearsonr(g, p)[0]
            sr = spearmanr(g, p)[0]

        per_emotion_pearson.append(pr)
        per_emotion_spearman.append(sr)

    if emo_list is None:
        emo_list = [f"Emotion {i}" for i in range(all_golds.shape[1])]

    print("Per-emotion Pearson:")
    for emo, v in zip(emo_list, per_emotion_pearson):
        print(f"  {emo}: {v:.4f}" if not np.isnan(v) else f"  {emo}: nan")

    print("Per-emotion Spearman:")
    for emo, v in zip(emo_list, per_emotion_spearman):
        print(f"  {emo}: {v:.4f}" if not np.isnan(v) else f"  {emo}: nan")

    return total_loss / max(1, n), pearson_all, spearman_all

In [ ]:
from scipy.stats import pearsonr, spearmanr

def data_processing_vanilla(train_s, dev_s, test_s, tokenizer, max_length=128):
    return (UtteranceOnlyDataset(train_s, tokenizer, max_length),
            UtteranceOnlyDataset(dev_s,   tokenizer, max_length),
            UtteranceOnlyDataset(test_s,  tokenizer, max_length))


def data_processing_srl(train_s, dev_s, test_s, tokenizer, max_length=128):
    label2id = {}
    for s in train_s + dev_s:
        for fr in s.frames:
            if fr and fr.labels:
                for l in fr.labels: label2id.setdefault(l, len(label2id))
    return (SRLDataset(train_s, tokenizer, label2id, max_length),
            SRLDataset(dev_s,   tokenizer, label2id, max_length),
            SRLDataset(test_s,  tokenizer, label2id, max_length),
            label2id)


def run_training(model, train_loader, dev_loader, device,
                 num_epochs, save_path, hist_path, model_key,
                 emo_list=None, grad_accum_steps=1, lr=5e-5):
    """Generic training loop — saves best checkpoint by dev Spearman."""

    optimizer   = torch.optim.AdamW(model.parameters(), lr=lr)
    total_steps = len(train_loader) * num_epochs // max(1, grad_accum_steps)
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        int(0.1 * total_steps),
        total_steps
    )

    history = {
        "epoch": [],
        "train_loss": [],
        "dev_loss": [],
        "dev_pearson": [],
        "dev_spearman": [],
    }

    best_spearman = -1.0

    print(f"\n{'='*55}\nTraining: {model_key}  ({num_epochs} epochs)\n{'='*55}")

    for epoch in range(num_epochs):
        tr_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            device,
            scheduler=scheduler,
            grad_accum_steps=grad_accum_steps
        )

        dev_loss, dev_pearson, dev_spearman = eval_emotion(
            model,
            dev_loader,
            device,
            emo_list=emo_list
        )

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(tr_loss)
        history["dev_loss"].append(dev_loss)
        history["dev_pearson"].append(dev_pearson)
        history["dev_spearman"].append(dev_spearman)

        print(f"Epoch {epoch+1:>2}: train_loss={tr_loss:.4f}  dev_loss={dev_loss:.4f}"
              f"  Pearson={dev_pearson:.4f}  Spearman={dev_spearman:.4f}")

        if dev_spearman > best_spearman:
            best_spearman = dev_spearman

            torch.save({
                "model_state": model.state_dict(),
                "best_dev_spearman": best_spearman,
                "hparams": {"model_key": model_key}
            }, save_path)
            print("  ↑ new best saved")

    with open(hist_path, "wb") as f:
        pickle.dump(history, f)

    print(f"History → {hist_path}")
    print("Best dev Spearman:", best_spearman)
    return history


## EMA Datatset

In [ ]:
from datetime import datetime
BASE_PATH = "/Enter-your-path/final_dataset_only_ema1/"
MODEL_LOG = "/Enter-your-path/ema1_finetune/"
HIST_LOG  = MODEL_LOG + "loss_history/"
os.makedirs(MODEL_LOG, exist_ok=True)
os.makedirs(HIST_LOG,  exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
# today  = date.today().strftime("%m-%d-%Y")
today = datetime.now().strftime("%m-%d-%Y_%H-%M-%S")
# print(BASE_PATH)

train_samples = create_utterance_samples(BASE_PATH + "srl_ema_ft_train.aligned.jsonl")
dev_samples   = create_utterance_samples(BASE_PATH + "srl_ema_ft_dev.aligned.jsonl")
test_samples  = create_utterance_samples(BASE_PATH + "srl_ema_ft_test.aligned.jsonl")
print(f"Train: {len(train_samples)} | Dev: {len(dev_samples)} | Test: {len(test_samples)}")

# 5. Training: Model 1 — Vanilla BERT

### Vanilla nomlp

In [ ]:
# BERT_NAME = "bert-base-cased"
# MODEL_KEY = "vanilla_BERT"
# tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)
# pad_id    = tokenizer.pad_token_id

# train_ds, dev_ds, test_ds = data_processing_vanilla(train_samples, dev_samples, test_samples, tokenizer)
# collate      = lambda b: utterance_collate(b, pad_id)
# train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  collate_fn=collate)
# dev_loader   = DataLoader(dev_ds,   batch_size=16, shuffle=False, collate_fn=collate)

# model = VanillaEmotionClassifier(BERT_NAME).to(device)
# run_training(model, train_loader, dev_loader, device, num_epochs=12,
#              save_path=MODEL_LOG + f"{today}_best_{MODEL_KEY}.ckpt",
#              hist_path=HIST_LOG  + f"{today}_{MODEL_KEY}.pkl", model_key=MODEL_KEY)


#vanila no mlp
BERT_NAME = "bert-base-uncased"
MODEL_KEY = "vanilla_BERT_macro_28emo"
tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)
pad_id    = tokenizer.pad_token_id

from datetime import datetime

now = datetime.now()
today = now.strftime("%Y-%m-%d_%H-%M-%S")

train_ds, dev_ds, test_ds = data_processing_vanilla(train_samples, dev_samples, test_samples, tokenizer)
collate      = lambda b: utterance_collate(b, pad_id)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  collate_fn=collate)
dev_loader   = DataLoader(dev_ds,   batch_size=16, shuffle=False, collate_fn=collate)

model = VanillaEmotionClassifier(BERT_NAME, num_emotions=NUM_EMOTIONS).to(device)
run_training(model, train_loader, dev_loader, device, num_epochs=10,
             save_path=MODEL_LOG + f"{today}_best_{MODEL_KEY}.ckpt",
             hist_path=HIST_LOG  + f"{today}_{MODEL_KEY}.pkl", model_key=MODEL_KEY)

### Vanilla mlp

In [ ]:
#vanila no mlp
BERT_NAME = "bert-base-uncased"
MODEL_KEY = "vanilla_BERT_macro_28emo_mlp"
tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)
pad_id    = tokenizer.pad_token_id

from datetime import datetime

now = datetime.now()
today = now.strftime("%Y-%m-%d_%H-%M-%S")

train_ds, dev_ds, test_ds = data_processing_vanilla(train_samples, dev_samples, test_samples, tokenizer)
collate      = lambda b: utterance_collate(b, pad_id)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  collate_fn=collate)
dev_loader   = DataLoader(dev_ds,   batch_size=16, shuffle=False, collate_fn=collate)

model = VanillaEmotionClassifier_mlp(BERT_NAME, num_emotions=NUM_EMOTIONS).to(device)
run_training(model, train_loader, dev_loader, device, num_epochs=5,
             save_path=MODEL_LOG + f"{today}_best_{MODEL_KEY}.ckpt",
             hist_path=HIST_LOG  + f"{today}_{MODEL_KEY}.pkl", model_key=MODEL_KEY)

# 6. Model 2 — SRL-Aware BERT (2args + Predicate)

### no fusion nomlp

In [ ]:

#ARG1 ARGM, Pred

BERT_NAME = "bert-base-uncased"
MODEL_KEY = "SRL_BERT_arg1_m_pred_macro"
tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)
pad_id    = tokenizer.pad_token_id

train_ds, dev_ds, test_ds, label2id = data_processing_srl(train_samples, dev_samples, test_samples, tokenizer)
collate      = lambda b: srl_collate_ulevel(b, pad_id)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  collate_fn=collate)
dev_loader   = DataLoader(dev_ds,   batch_size=16, shuffle=False, collate_fn=collate)

model = SRLEmotionClassifierAttention_nofusion_three_nomlp(BERT_NAME, num_labels=len(label2id)).to(device)
run_training(model, train_loader, dev_loader, device, num_epochs=5,
             
             save_path=MODEL_LOG + f"{today}_best_{MODEL_KEY}.ckpt",
             hist_path=HIST_LOG  + f"{today}_{MODEL_KEY}.pkl", model_key=MODEL_KEY)

### nofusion mlp

In [ ]:
now = datetime.now()
today = now.strftime("%Y-%m-%d_%H-%M-%S")

BERT_NAME = "bert-base-uncased"
MODEL_KEY = "SRL_BERT_arg1_m_pred_Attention_macro_emo28_nofusion_mlp"
tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)
pad_id    = tokenizer.pad_token_id

train_ds, dev_ds, test_ds, label2id = data_processing_srl(train_samples, dev_samples, test_samples, tokenizer)
collate      = lambda b: srl_collate_ulevel(b, pad_id)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  collate_fn=collate)
dev_loader   = DataLoader(dev_ds,   batch_size=16, shuffle=False, collate_fn=collate)

model = SRLEmotionClassifierAttention_nofusion_three_mlp(BERT_NAME,num_emotions=NUM_EMOTIONS, num_labels=len(label2id)).to(device)
run_training(model, train_loader, dev_loader, device, num_epochs=5,
             save_path=MODEL_LOG + f"{today}_best_{MODEL_KEY}.ckpt",
             hist_path=HIST_LOG  + f"{today}_{MODEL_KEY}.pkl", model_key=MODEL_KEY)

### fusion no mlp

In [ ]:



# #ARG1 ARGM, Pred, attention

# BERT_NAME = "bert-base-cased"
# MODEL_KEY = "SRL_BERT_arg1_m_pred_LSTM_Attention"
# tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)
# pad_id    = tokenizer.pad_token_id

# train_ds, dev_ds, test_ds, label2id = data_processing_srl(train_samples, dev_samples, test_samples, tokenizer)
# collate      = lambda b: srl_collate_ulevel(b, pad_id)
# train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  collate_fn=collate)
# dev_loader   = DataLoader(dev_ds,   batch_size=16, shuffle=False, collate_fn=collate)

# model = SRLEmotionClassifierAttention_three(BERT_NAME, num_labels=len(label2id)).to(device)
# run_training(model, train_loader, dev_loader, device, num_epochs=12,
#              save_path=MODEL_LOG + f"{today}_best_{MODEL_KEY}.ckpt",
#              hist_path=HIST_LOG  + f"{today}_{MODEL_KEY}.pkl", model_key=MODEL_KEY)





#ARG1 ARGM, Pred, attention, macro

now = datetime.now()
today = now.strftime("%Y-%m-%d_%H-%M-%S")

BERT_NAME = "bert-base-uncased"
MODEL_KEY = "SRL_BERT_arg1_m_pred_Attention_macro_emo28_fusion"
tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)
pad_id    = tokenizer.pad_token_id

train_ds, dev_ds, test_ds, label2id = data_processing_srl(train_samples, dev_samples, test_samples, tokenizer)
collate      = lambda b: srl_collate_ulevel(b, pad_id)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  collate_fn=collate)
dev_loader   = DataLoader(dev_ds,   batch_size=16, shuffle=False, collate_fn=collate)

model = SRLEmotionClassifierAttention_three(BERT_NAME,num_emotions=NUM_EMOTIONS, num_labels=len(label2id)).to(device)
run_training(model, train_loader, dev_loader, device, num_epochs=5,
             save_path=MODEL_LOG + f"{today}_best_{MODEL_KEY}.ckpt",
             hist_path=HIST_LOG  + f"{today}_{MODEL_KEY}.pkl", model_key=MODEL_KEY)

### fusion mlp

In [ ]:
now = datetime.now()
today = now.strftime("%Y-%m-%d_%H-%M-%S")

BERT_NAME = "bert-base-uncased"
MODEL_KEY = "SRL_BERT_arg1_m_pred_Attention_macro_emo28_fusion_mlp"
tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)
pad_id    = tokenizer.pad_token_id

train_ds, dev_ds, test_ds, label2id = data_processing_srl(train_samples, dev_samples, test_samples, tokenizer)
collate      = lambda b: srl_collate_ulevel(b, pad_id)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  collate_fn=collate)
dev_loader   = DataLoader(dev_ds,   batch_size=16, shuffle=False, collate_fn=collate)

model = SRLEmotionClassifierAttention_fusion_three_mlp(BERT_NAME,num_emotions=NUM_EMOTIONS, num_labels=len(label2id)).to(device)
run_training(model, train_loader, dev_loader, device, num_epochs=5,
             save_path=MODEL_LOG + f"{today}_best_{MODEL_KEY}.ckpt",
             hist_path=HIST_LOG  + f"{today}_{MODEL_KEY}.pkl", model_key=MODEL_KEY)

## 12. Test Evaluation

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, f1_score
import pandas as pd
import numpy as np


with open("./GoEmotion/emotions.txt") as f:
    EMOTION_NAMES = [l.strip() for l in f]
    
SENTIMENT_MAP = {
"positive": ["amusement", "excitement", "joy", "love", "desire", "optimism", "caring", "pride", "admiration", "gratitude", "relief", "approval"],
"negative": ["fear", "nervousness", "remorse", "embarrassment", "disappointment", "sadness", "grief", "disgust", "anger", "annoyance", "disapproval"],
"ambiguous": ["realization", "surprise", "curiosity", "confusion"], 
    "neutral":["neutral"]
}

# build emotion->index
EMO2IDX = {e: i for i, e in enumerate(EMOTION_NAMES)}

# sentiment -> indices (ignore missing emotions safely)
SENT2IDXS = {
    s: [EMO2IDX[e] for e in emos if e in EMO2IDX]
    for s, emos in SENTIMENT_MAP.items()
}
SENTIMENT_ORDER = ["positive", "negative", "ambiguous"]



def sentiment_metrics(P, G, sent2idxs, order=("positive", "negative", "ambiguous")):
    """
    P, G: numpy arrays [N, E] with 0/1
    returns:
      sent_P, sent_G: [N, 3]
      sent_df: list of dict rows with precision/recall/f1/support
      summary: dict with micro/macro F1 over 3 sentiments
    """
    sent_P = np.zeros((P.shape[0], len(order)), dtype=int)
    sent_G = np.zeros((G.shape[0], len(order)), dtype=int)

    for j, s in enumerate(order):
        idxs = sent2idxs.get(s, [])
        if len(idxs) == 0:
            continue
        sent_P[:, j] = (P[:, idxs].sum(axis=1) > 0).astype(int)
        sent_G[:, j] = (G[:, idxs].sum(axis=1) > 0).astype(int)

    prec, rec, f1, sup = precision_recall_fscore_support(
        sent_G, sent_P, average=None, zero_division=0
    )

    rows = []
    for j, s in enumerate(order):
        rows.append({
            "sentiment": s,
            "precision": float(prec[j]),
            "recall": float(rec[j]),
            "f1": float(f1[j]),
            "support": int(sup[j]),
        })

    summary = {
        "micro_f1": float(f1_score(sent_G, sent_P, average="micro", zero_division=0)),
        "macro_f1": float(f1_score(sent_G, sent_P, average="macro", zero_division=0)),
    }
    return sent_P, sent_G, rows, summary


@torch.no_grad()
def test_emotion(model, dataloader, device, threshold=THRESHOLD, emotion_names=None):
    model.eval()
    all_preds, all_golds = [], []
    total_loss, n = 0.0, 0

    for batch in dataloader:
        batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
        logits, loss, _ = model(**batch)
        total_loss += float(loss); n += 1

        if "emotion_labels" in batch:
            probs = torch.sigmoid(logits)
            all_preds.append((probs >= threshold).float().cpu())
            all_golds.append(batch["emotion_labels"].float().cpu())

    P = torch.cat(all_preds).numpy()  # [num_examples, E]
    G = torch.cat(all_golds).numpy()  # [num_examples, E]

    # --- overall micro/macro F1 ---
    f1_micro = f1_score(G, P, average="micro", zero_division=0)
    f1_macro = f1_score(G, P, average="macro", zero_division=0)

    # --- per-emotion precision/recall/f1/support ---
    prec, rec, f1, support = precision_recall_fscore_support(
        G, P, average=None, zero_division=0
    )
    
    prec = np.round(prec, 3)
    rec  = np.round(rec, 3)
    f1   = np.round(f1, 3)

    if emotion_names is None:
        emotion_names = [f"emotion_{i}" for i in range(G.shape[1])]

    per_emotion_df = pd.DataFrame({
        "emotion": emotion_names,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "support": support.astype(int),
    })

    # <<< your sentiment-level PRF (unchanged)
    sent_P, sent_G, sent_rows, sent_summary = sentiment_metrics(
        P, G, SENT2IDXS, order=SENTIMENT_ORDER
    )

    print("G shape:", G.shape)
    print("P shape:", P.shape)
    print("len(emotion_names):", len(emotion_names) if emotion_names is not None else None)
    print("len(prec):", len(prec))
    print("len(rec):", len(rec))
    print("len(f1):", len(f1))
    print("len(support):", len(support))
    
    return (
        total_loss / max(1, n),
        f1_micro,
        f1_macro,
        per_emotion_df,   # <<< CHANGED (DF instead of f1_pc array)
        P, G,
        sent_rows, sent_summary
    )


def run_test_experiment(exp, test_samples, device):
    tokenizer = AutoTokenizer.from_pretrained(exp["bert_name"])
    pad_id    = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    ckpt      = torch.load(exp["ckpt_path"], map_location=device, weights_only=False)

    if exp["architecture"] == "Vanilla-nomlp":
        model   = VanillaEmotionClassifier(exp["bert_name"], dropout=0.0).to(device)
        test_ds = UtteranceOnlyDataset(test_samples, tokenizer)
        collate = lambda b: utterance_collate(b, pad_id)

    elif exp["architecture"] == "Vanilla-mlp":
        model   = VanillaEmotionClassifier_mlp(exp["bert_name"], dropout=0.0).to(device)
        test_ds = UtteranceOnlyDataset(test_samples, tokenizer)
        collate = lambda b: utterance_collate(b, pad_id)

    elif exp["architecture"] == "nofusion-nomlp":
        label2id = ckpt.get("label2id", {"O": 0})
        model    = SRLEmotionClassifierAttention_nofusion_three_nomlp(exp["bert_name"], num_labels=len(label2id), dropout=0.0).to(device)
        test_ds  = SRLDataset(test_samples, tokenizer, label2id)
        collate  = lambda b: srl_collate_ulevel(b, pad_id)

    elif exp["architecture"] == "nofusion-mlp":
        label2id = ckpt.get("label2id", {"O": 0})
        model    = SRLEmotionClassifierAttention_nofusion_three_mlp(exp["bert_name"], num_labels=len(label2id), dropout=0.0).to(device)
        test_ds  = SRLDataset(test_samples, tokenizer, label2id)
        collate  = lambda b: srl_collate_ulevel(b, pad_id)
        
    elif exp["architecture"] == "fusion-nomlp":
        label2id = ckpt.get("label2id", {"O": 0})
        model    = SRLEmotionClassifierAttention_fusion_three_nomlp(exp["bert_name"], num_labels=len(label2id), dropout=0.0).to(device)
        test_ds  = SRLDataset(test_samples, tokenizer, label2id)
        collate  = lambda b: srl_collate_ulevel(b, pad_id)
        
    elif exp["architecture"] == "fusion-mlp":
        label2id = ckpt.get("label2id", {"O": 0})
        model    = SRLEmotionClassifierAttention_fusion_three_mlp(exp["bert_name"], num_labels=len(label2id), dropout=0.0).to(device)
        test_ds  = SRLDataset(test_samples, tokenizer, label2id)
        collate  = lambda b: srl_collate_ulevel(b, pad_id)

    else: 
        raise ValueError(f"Unknown architecture: {exp['architecture']}")

    model.load_state_dict(ckpt["model_state"], strict=False)
    model.eval()

    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=collate)

    loss, f1_micro, f1_macro, per_emotion_df, preds, golds, sent_rows, sent_summary = test_emotion(
        model, test_loader, device, threshold=THRESHOLD, emotion_names=EMOTION_NAMES
    )

    print(f"\n[{exp['name']}]  loss={loss:.4f}  F1_micro={f1_micro:.4f}  F1_macro={f1_macro:.4f}")


    # per-emotion PRF
    pc_df = per_emotion_df.sort_values("f1", ascending=False)
    print(pc_df.to_string(index=False))

    # sentiment-level PRF (your existing)
#     print("\nSentiment-level (OR over emotions in each group):")
#     print(f"  micro_F1={sent_summary['micro_f1']:.4f}  macro_F1={sent_summary['macro_f1']:.4f}")
#     sent_df = pd.DataFrame(sent_rows).sort_values("f1", ascending=False)
#     print(sent_df.to_string(index=False))
    
    now = datetime.now()
    today = now.strftime("%Y-%m-%d_%H-%M-%S")
    
    safe_name = exp["name"].replace(" ", "_").replace("/", "_")
    save_path = f"{MODEL_LOG}{safe_name}_{today}_per_emotion_prf.csv"
    
        
    # --- Save per-example predictions & golds for significance testing ---
    pg_dir = os.path.join(MODEL_LOG, "pred_gold")
    os.makedirs(pg_dir, exist_ok=True)

    pg_path = os.path.join(pg_dir, f"{safe_name}_{today}_P_G.npz")
    np.savez_compressed(
        pg_path,
        P=preds.astype(np.int8),
        G=golds.astype(np.int8),
        emotion_names=np.array(EMOTION_NAMES, dtype=object),
        model_name=np.array(exp["name"], dtype=object),
        threshold=np.array([THRESHOLD], dtype=float),
    )
    print(f"Saved P/G arrays → {pg_path}")

    per_emotion_df.to_csv(save_path, index=False)
    print(f"Saved per-emotion PRF → {save_path}")

    # optionally return everything so you can save/compare later
    return {
        "name": exp["name"],
        "loss": loss,
        "f1_micro": f1_micro,
        "f1_macro": f1_macro,
        "P": preds,
        "G": golds,
        "per_emotion_df": per_emotion_df,
        "sentiment_df": pd.DataFrame(sent_rows),
        "sent_summary": sent_summary,
    }

## For Goemotion data

In [ ]:
MODEL_LOG = "./model_log/"
EXPERIMENTS = [
    {"name": "Vanilla BERT",      "bert_name": "bert-base-uncased", "architecture": "Vanilla",
     "ckpt_path": MODEL_LOG + "2026-04-21_19-27-17_best_vanilla_BERT_macro_28emo.ckpt"}
    ,{"name": "SRL BERT_Attention",     "bert_name": "bert-base-uncased", "architecture": "SRL_Attention",
     "ckpt_path": MODEL_LOG + "macro_f1/2026-04-23_13-04-56_best_SRL_BERT_arg1_m_pred_Attention_macro_emo28_fusion.ckpt"}
    
]

# /Enter-your-path/2026-04-23_13-04-56_best_SRL_BERT_arg1_m_pred_Attention_macro_emo28_fusion.ckpt
# /Enter-your-path/2026-04-21_19-27-17_best_vanilla_BERT_macro_28emo.ckpt
# best: "2026-04-21_23-40-36_best_SRL_BERT_arg1_m_pred_Attention_macro_emo28.ckpt"

now = datetime.now()
today = now.strftime("%Y-%m-%d_%H-%M-%S")

with open("./GoEmotion/emotions.txt") as f:
    EMOTION_NAMES = [l.strip() for l in f]

summary = []
all_emotion_rows = []   # collect per-emotion PRF from all experiments
all_sent_rows = []      # collect sentiment PRF from all experiments (optional)
all_P_G = []

for exp in EXPERIMENTS:
    if not os.path.exists(exp["ckpt_path"]):
        print(f"[SKIP] {exp['name']} — checkpoint not found")
        continue

    res = run_test_experiment(exp, test_samples, device)  # <-- uses updated version
    summary.append(res)

    # --- collect per-emotion dataframe with model name attached ---
    per_df = res["per_emotion_df"].copy()
    per_df.insert(0, "model", res["name"])
    all_emotion_rows.append(per_df)

    # --- collect sentiment-level dataframe (optional) ---
    sent_df = res["sentiment_df"].copy()
    sent_df.insert(0, "model", res["name"])
    all_sent_rows.append(sent_df)
    
    all_P_G.append((res["name"], res["P"], res["G"]))
    # (optional) save per-model detailed outputs
    # safe_name = res["name"].replace(" ", "_").replace("/", "_")
    # per_df.to_csv(f"{MODEL_LOG}{safe_name}_per_emotion_prf.csv", index=False)
    # sent_df.to_csv(f"{MODEL_LOG}{safe_name}_sentiment_prf.csv", index=False)

# -------------------------
# Summary table print
# -------------------------
print(f"\n{'='*68}")
print(f"{'Model':<22} | {'F1 Micro':>9} | {'F1 Macro':>9} | {'Loss':>8}")
print("-" * 68)
for r in summary:
    print(f"{r['name']:<22} | {r['f1_micro']:>9.4f} | {r['f1_macro']:>9.4f} | {r['loss']:>8.4f}")

# -------------------------
# Combined DataFrames
# -------------------------
if len(all_emotion_rows) > 0:
    all_emotions_df = pd.concat(all_emotion_rows, ignore_index=True)

    # Example: sort by model then by f1 descending
    all_emotions_df = all_emotions_df.sort_values(["model", "f1"], ascending=[True, False])

    print("\n[Combined per-emotion PRF] (head)")
    print(all_emotions_df.head(30).to_string(index=False))

    # (optional) save combined
    all_emotions_df.to_csv(f"{MODEL_LOG}ALL_MODELS_per_emotion_prf_{today}.csv", index=False)

# if len(all_sent_rows) > 0:
#     all_sentiments_df = pd.concat(all_sent_rows, ignore_index=True)
#     all_sentiments_df = all_sentiments_df.sort_values(["model", "f1"], ascending=[True, False])

#     print("\n[Combined sentiment PRF]")
#     print(all_sentiments_df.to_string(index=False))

#     # (optional) save combined
#     all_sentiments_df.to_csv(f"{MODEL_LOG}_ARG1_ARGM_ALL_MODELS_sentiment_prf.csv", index=False)

In [ ]:
MODEL_LOG = "./model_log/current_best/"
EXPERIMENTS = [
    {"name": "Vanilla-nomlp",      "bert_name": "bert-base-uncased", "architecture": "Vanilla-nomlp",
     "ckpt_path": MODEL_LOG + "2026-04-21_19-27-17_best_vanilla_BERT_macro_28emo_nomlp.ckpt"}
    
    ,{"name": "Vanilla-mlp",     "bert_name": "bert-base-uncased", "architecture": "Vanilla-mlp",
     "ckpt_path": MODEL_LOG + "2026-04-26_21-57-07_best_vanilla_BERT_macro_28emo_mlp.ckpt"},    
    
    {"name": "nofusion-nomlp",      "bert_name": "bert-base-uncased", "architecture": "nofusion-nomlp",
     "ckpt_path": MODEL_LOG + "2026-04-21_23-40-36_best_SRL_BERT_arg1_m_pred_Attention_macro_emo28_nofusion_nomlp.ckpt"}
    
    ,{"name": "nofusion-mlp",     "bert_name": "bert-base-uncased", "architecture": "nofusion-mlp",
     "ckpt_path": MODEL_LOG + "2026-04-26_23-05-42_best_SRL_BERT_arg1_m_pred_Attention_macro_emo28_nofusion_mlp.ckpt"},    
    
    {"name": "fusion-nomlp",      "bert_name": "bert-base-uncased", "architecture": "fusion-nomlp",
     "ckpt_path": MODEL_LOG + "2026-04-23_13-04-56_best_SRL_BERT_arg1_m_pred_Attention_macro_emo28_fusion_nomlp.ckpt"}
    
    ,{"name": "fusion-mlp",     "bert_name": "bert-base-uncased", "architecture": "fusion-mlp",
     "ckpt_path": MODEL_LOG + "2026-04-26_22-03-51_best_SRL_BERT_arg1_m_pred_Attention_macro_emo28_fusion_mlp.ckpt"}
    
    
    
]

# /Enter-your-path/2026-04-23_13-04-56_best_SRL_BERT_arg1_m_pred_Attention_macro_emo28_fusion.ckpt
# /Enter-your-path/2026-04-21_19-27-17_best_vanilla_BERT_macro_28emo.ckpt
# best: "2026-04-21_23-40-36_best_SRL_BERT_arg1_m_pred_Attention_macro_emo28.ckpt"

now = datetime.now()
today = now.strftime("%Y-%m-%d_%H-%M-%S")

with open("./GoEmotion/emotions.txt") as f:
    EMOTION_NAMES = [l.strip() for l in f]

summary = []
all_emotion_rows = []   # collect per-emotion PRF from all experiments
all_sent_rows = []      # collect sentiment PRF from all experiments (optional)
all_P_G = []

for exp in EXPERIMENTS:
    if not os.path.exists(exp["ckpt_path"]):
        print(f"[SKIP] {exp['name']} — checkpoint not found")
        continue

    res = run_test_experiment(exp, test_samples, device)  # <-- uses updated version
    summary.append(res)

    # --- collect per-emotion dataframe with model name attached ---
    per_df = res["per_emotion_df"].copy()
    per_df.insert(0, "model", res["name"])
    all_emotion_rows.append(per_df)

    # --- collect sentiment-level dataframe (optional) ---
    sent_df = res["sentiment_df"].copy()
    sent_df.insert(0, "model", res["name"])
    all_sent_rows.append(sent_df)
    
    all_P_G.append((res["name"], res["P"], res["G"]))
    # (optional) save per-model detailed outputs
    # safe_name = res["name"].replace(" ", "_").replace("/", "_")
    # per_df.to_csv(f"{MODEL_LOG}{safe_name}_per_emotion_prf.csv", index=False)
    # sent_df.to_csv(f"{MODEL_LOG}{safe_name}_sentiment_prf.csv", index=False)

# -------------------------
# Summary table print
# -------------------------
print(f"\n{'='*68}")
print(f"{'Model':<22} | {'F1 Micro':>9} | {'F1 Macro':>9} | {'Loss':>8}")
print("-" * 68)
for r in summary:
    print(f"{r['name']:<22} | {r['f1_micro']:>9.4f} | {r['f1_macro']:>9.4f} | {r['loss']:>8.4f}")

# -------------------------
# Combined DataFrames
# -------------------------
if len(all_emotion_rows) > 0:
    all_emotions_df = pd.concat(all_emotion_rows, ignore_index=True)

    # Example: sort by model then by f1 descending
    all_emotions_df = all_emotions_df.sort_values(["model", "f1"], ascending=[True, False])

    print("\n[Combined per-emotion PRF] (head)")
    print(all_emotions_df.head(30).to_string(index=False))

    # (optional) save combined
    all_emotions_df.to_csv(f"{MODEL_LOG}ALL_MODELS_per_emotion_prf_{today}.csv", index=False)

# if len(all_sent_rows) > 0:
#     all_sentiments_df = pd.concat(all_sent_rows, ignore_index=True)
#     all_sentiments_df = all_sentiments_df.sort_values(["model", "f1"], ascending=[True, False])

#     print("\n[Combined sentiment PRF]")
#     print(all_sentiments_df.to_string(index=False))

#     # (optional) save combined
#     all_sentiments_df.to_csv(f"{MODEL_LOG}_ARG1_ARGM_ALL_MODELS_sentiment_prf.csv", index=False)